## Build the Model Specifications

In [4]:
from orchestra import Sequential
from orchestra.arch import Dense
from orchestra.initialization import Kaiming
from orchestra.activations import Sigmoid

model = Sequential([
    Dense(2, Kaiming(), Sigmoid()),
    Dense(1, Kaiming(), Sigmoid()),
])

## Build the Training Specifications

In [5]:
import orchestra
from orchestra.datasets import InlineDataset
from orchestra.optimizers import GradientDescent
from orchestra.sync import BarrierSync
from orchestra.store import BlockingStore

N_WORKERS = 3
N_SERVERS = 2

xor_dataset = [
    0.0, 0.0, 0.0,
    0.0, 1.0, 1.0,
    1.0, 0.0, 1.0,
    1.0, 1.0, 0.0,
]

training = orchestra.parameter_server(
    worker_addrs=[f"worker-{i}:{50_000 + i}" for i in range(N_WORKERS)],
    server_addrs=[f"server-{i}:{40_000 + i}" for i in range(N_SERVERS)],
    dataset=InlineDataset(xor_dataset, x_size=2, y_size=1),
    optimizer=GradientDescent(lr=1.0),
    sync=BarrierSync(),
    store=BlockingStore(),
    max_epochs=500,
    batch_size=4,
    offline_epochs=0,
    seed=42,
)

## Train the model

In [6]:
import orchestra

session = orchestra.orchestrate(model, training)
trained_model = session.wait()

  epoch 1/500 avg_loss=0.25762096
  epoch 1/500 avg_loss=0.30057156
  epoch 1/500 avg_loss=0.25727651
  epoch 2/500 avg_loss=0.24161291
  epoch 2/500 avg_loss=0.23988873
  epoch 2/500 avg_loss=0.25184014
  epoch 3/500 avg_loss=0.25138667
  epoch 3/500 avg_loss=0.24343896
  epoch 3/500 avg_loss=0.25031108
  epoch 4/500 avg_loss=0.24637973
  epoch 4/500 avg_loss=0.24998032
  epoch 4/500 avg_loss=0.24989097
  epoch 5/500 avg_loss=0.24793513
  epoch 5/500 avg_loss=0.24974950
  epoch 5/500 avg_loss=0.24975567
  epoch 6/500 avg_loss=0.24876796
  epoch 6/500 avg_loss=0.24965972
  epoch 6/500 avg_loss=0.24969101
  epoch 7/500 avg_loss=0.24917984
  epoch 7/500 avg_loss=0.24960439
  epoch 7/500 avg_loss=0.24964301
  epoch 8/500 avg_loss=0.24936712
  epoch 8/500 avg_loss=0.24955674
  epoch 8/500 avg_loss=0.24959825
  epoch 9/500 avg_loss=0.24943821
  epoch 9/500 avg_loss=0.24950983
  epoch 9/500 avg_loss=0.24955319
  epoch 10/500 avg_loss=0.24944980
  epoch 10/500 avg_loss=0.24949487
  epoch 10/5

## Save the model

In [8]:
MODEL_FILE_PATH = "./xor.safetensors"

trainer_model.save_safetensors(MODEL_FILE_PATH)

## Pytorch Eval

In [12]:
import torch
import torch.nn as nn
from safetensors.torch import load_file


class Xor(nn.Module):
    def __init__(self, input_size: int, architecture: list[int]):
        super().__init__()
        sizes = [input_size] + architecture
        self.linears = nn.ModuleList([
            nn.Linear(sizes[i], sizes[i+1])
            for i in range(len(architecture))
        ])
        self.acts = [torch.sigmoid for _ in architecture]

    def forward(self, x):
        for linear, act in zip(self.linears, self.acts):
            x = act(linear(x))

        return x


state_dict = load_file(MODEL_FILE_PATH)
model = Xor(2, [2, 1])

In [29]:
with torch.no_grad():
    for i, linear in enumerate(model.linears):
        linear.weight.copy_(state_dict[f"layer_{i}.weight"].T)
        linear.bias.copy_(state_dict[f"layer_{i}.bias"])

model.eval()

x = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])

y = torch.tensor([
    [0.0],
    [1.0],
    [1.0],
    [0.0],
])

with torch.no_grad():
    preds = model(x) >= 0.5
    acc = (preds == y).float().mean().item() * 100

print(f"\nTest accuracy: {acc:.2f}%")


Test accuracy: 100.00%
